In [10]:
import numpy as np
import pandas as pd

from scipy import sparse
from sklearn.preprocessing import normalize
from sklearn.decomposition import TruncatedSVD
from sklearn.cluster import KMeans


RANDOM_STATE = 42

N_CLUSTERS_LIST = [35, 40, 45]
N_COMPONENTS_LIST = [100, 200, 300, 500, 1000]


# =========================
# 1. Загрузка данных
# =========================

train = sparse.load_npz("train.npz").astype(np.float32)

X = train.copy()

X.data = np.nan_to_num(
    X.data,
    nan=0.0,
    posinf=0.0,
    neginf=0.0
)

X.eliminate_zeros()

print("Train shape:", X.shape)
print("Min:", X.data.min())
print("Max:", X.data.max())
print("Has NaN:", np.isnan(X.data).any())
print("Has inf:", np.isinf(X.data).any())


# =========================
# 2. L2-нормализация исходной sparse-матрицы
# =========================

X = normalize(X, norm="l2", axis=1)


# =========================
# 3. Функция сохранения submission
# =========================

def save_submission(labels, filename):
    submission = pd.DataFrame({
        "ID": np.arange(len(labels)),
        "TARGET": labels.astype(int)
    })

    submission.to_csv(filename, index=False)
    print("saved:", filename)


# =========================
# 4. KMeans++ для разных SVD и k
# =========================

for n_components in N_COMPONENTS_LIST:
    print("\n" + "=" * 80)
    print(f"SVD components: {n_components}")
    print("=" * 80)

    svd = TruncatedSVD(
        n_components=n_components,
        random_state=RANDOM_STATE
    )

    X_svd = svd.fit_transform(X)

    X_svd = np.nan_to_num(
        X_svd,
        nan=0.0,
        posinf=0.0,
        neginf=0.0
    )

    explained = svd.explained_variance_ratio_.sum()
    print(f"Explained variance: {explained:.5f}")

    # Нормализация после SVD.
    # Это делает KMeans ближе к cosine-kmeans.
    X_svd = normalize(X_svd, norm="l2", axis=1)

    for n_clusters in N_CLUSTERS_LIST:
        print("\n" + "-" * 70)
        print(f"KMeans++: SVD={n_components}, clusters={n_clusters}")
        print("-" * 70)

        model = KMeans(
            n_clusters=n_clusters,
            init="k-means++",
            n_init=100,
            max_iter=500,
            random_state=RANDOM_STATE,
            verbose=0
        )

        labels = model.fit_predict(X_svd)

        save_submission(
            labels,
            f"submission_kmeanspp_svd{n_components}_k{n_clusters}.csv"
        )

Train shape: (21000, 3049)
Min: -253.84004
Max: 100000.0
Has NaN: False
Has inf: False

SVD components: 100
Explained variance: 0.86918

----------------------------------------------------------------------
KMeans++: SVD=100, clusters=35
----------------------------------------------------------------------
saved: submission_kmeanspp_svd100_k35.csv

----------------------------------------------------------------------
KMeans++: SVD=100, clusters=40
----------------------------------------------------------------------
saved: submission_kmeanspp_svd100_k40.csv

----------------------------------------------------------------------
KMeans++: SVD=100, clusters=45
----------------------------------------------------------------------
saved: submission_kmeanspp_svd100_k45.csv

SVD components: 200
Explained variance: 0.92359

----------------------------------------------------------------------
KMeans++: SVD=200, clusters=35
----------------------------------------------------------------

KeyboardInterrupt: 